In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import classification_report
import joblib
import optuna

warnings.filterwarnings('ignore')

# Load data from binary classification processed directory
X_train = pd.read_csv('../../dataset_processed/X_train_final.csv')
y_train = pd.read_csv('../../dataset_processed/y_train_final.csv').values.ravel()
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")


## Phase 1: Baseline Model

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['f1_macro', 'recall_macro', 'accuracy']

# Baseline Random Forest with balanced class weighting
baseline_model = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)

cv_results_base = cross_validate(baseline_model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
baseline_f1 = cv_results_base['test_f1_macro'].mean()

print(f"Mean CV Macro F1: {baseline_f1:.4f}")
print(f"Mean CV Macro Recall: {cv_results_base['test_recall_macro'].mean():.4f}")
print(f"Mean CV Accuracy: {cv_results_base['test_accuracy'].mean():.4f}")

baseline_model.fit(X_train, y_train)
y_pred_base = baseline_model.predict(X_train)
report_base = classification_report(y_train, y_pred_base)
print("\nClassification Report on Full Train Data (Baseline):\n", report_base)

os.makedirs('results', exist_ok=True)
with open('results/baseline_metrics.txt', 'w') as f:
    f.write(f"Mean CV Macro F1: {baseline_f1:.4f}\n")
    f.write(f"Mean CV Macro Recall: {cv_results_base['test_recall_macro'].mean():.4f}\n")
    f.write(f"Mean CV Accuracy: {cv_results_base['test_accuracy'].mean():.4f}\n\n")
    f.write("Classification Report on Full Train Data (Baseline):\n")
    f.write(report_base)


## Phase 2: Hyperparameter Tuning with Optuna

In [ ]:
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 100, 350)
    max_depth = trial.suggest_int('max_depth', 6, 18)
    min_samples_split = trial.suggest_int('min_samples_split', 4, 15)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 3, 12)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.4, 0.6])
    criterion = trial.suggest_categorical('criterion', ['gini', 'entropy'])
    max_samples = trial.suggest_float('max_samples', 0.7, 0.95)
    
    model = RandomForestClassifier(
        class_weight='balanced', 
        n_jobs=-1, 
        random_state=42, 
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        criterion=criterion,
        max_samples=max_samples
    )
    
    cv_results = cross_validate(model, X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)
    return cv_results['test_score'].mean()

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
print("Starting Random Forest Deep Hyperparameter Optimization (30 trials)...")
study.optimize(objective, n_trials=30, show_progress_bar=False)

print(f"\nBest Params: {study.best_params}")
print(f"Best Macro F1: {study.best_value:.4f}")


## Phase 3: Tuned Eval & Comparison

In [ ]:
tuned_model = RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=42, **study.best_params)

cv_results_tuned = cross_validate(tuned_model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
tuned_f1 = cv_results_tuned['test_f1_macro'].mean()

print(f"Mean CV Macro F1 (Tuned): {tuned_f1:.4f}")
print(f"Mean CV Macro Recall (Tuned): {cv_results_tuned['test_recall_macro'].mean():.4f}")
print(f"Mean CV Accuracy (Tuned): {cv_results_tuned['test_accuracy'].mean():.4f}")

tuned_model.fit(X_train, y_train)
y_pred_tuned = tuned_model.predict(X_train)
report_tuned = classification_report(y_train, y_pred_tuned)
print("\nClassification Report on Full Train Data (Tuned):\n", report_tuned)

os.makedirs('results', exist_ok=True)
with open('results/tuned_metrics.txt', 'w') as f:
    f.write(f"Mean CV Macro F1: {tuned_f1:.4f}\n")
    f.write(f"Mean CV Macro Recall: {cv_results_tuned['test_recall_macro'].mean():.4f}\n")
    f.write(f"Mean CV Accuracy: {cv_results_tuned['test_accuracy'].mean():.4f}\n\n")
    f.write("Classification Report on Full Train Data (Tuned):\n")
    f.write(report_tuned)

print("=== Random Forest Performance Comparison ===")
print(f"Baseline Macro F1: {baseline_f1:.4f}")
print(f"Tuned Macro F1:    {tuned_f1:.4f}")
print(f"Improvement:       {tuned_f1 - baseline_f1:+.4f}")


## Phase 4: Save Model

In [ ]:
if tuned_f1 > baseline_f1:
    print("Tuned model is better. Saving tuned model...")
    final_model = tuned_model
else:
    print("Baseline model is better or equal. Saving baseline model...")
    final_model = baseline_model

joblib.dump(final_model, 'random_forest_best_model.pkl')
print("Model saved successfully to random_forest_best_model.pkl")
